In [ ]:
# SQLAlchemy CRUD Operations

Working with PostgreSQL using SQLAlchemy ORM.

**Topics:**
- CREATE (INSERT)
- READ (SELECT)
- UPDATE
- DELETE
- Transactions (COMMIT & ROLLBACK)
- Error Handling

In [2]:
import os
import sys

project_path = r"C:\Users\yarag\Documents\GitHub\Food-Waste"

os.chdir(project_path)
sys.path.insert(0, project_path)

print("Current Directory:", os.getcwd())

Current Directory: C:\Users\yarag\Documents\GitHub\Food-Waste


In [3]:
from sqlalchemy import select
from sqlalchemy.orm import Session

from db.connection import engine
from db.models import (
    Country,
    Commodity,
    Method,
    Activity,
    Source,
    Observation,
    ObservationActivity
)

print("SQLAlchemy imports completed successfully!")

SQLAlchemy imports completed successfully!


In [ ]:
## 1. Test Database Connection

Verify that SQLAlchemy is successfully connected to the PostgreSQL database.

In [4]:
from sqlalchemy import text

with engine.connect() as connection:
    result = connection.execute(
        text("SELECT current_database();")
    )

    print("Connected successfully!")
    print("Database:", result.scalar())

Connected successfully!
Database: food_waste_db


In [ ]:
## 2. CREATE – Insert Records

Insert a new test country into the database using SQLAlchemy ORM.

We will use `session.add()` to add the record and `session.commit()` to save it permanently.

In [5]:
with Session(engine) as session:

    new_country = Country(
        m49_code="995",
        country_name="Notebook Test Country"
    )

    session.add(new_country)
    session.commit()

    print("Country inserted successfully!")
    print(f"Code: {new_country.m49_code}")
    print(f"Name: {new_country.country_name}")

Country inserted successfully!
Code: 995
Name: Notebook Test Country


In [ ]:
### 2.1 Verify Inserted Record

Retrieve the inserted country using its primary key to confirm that it exists in the database.

In [6]:
with Session(engine) as session:

    country = session.get(Country, "995")

    if country:
        print("Country found!")
        print(f"Code: {country.m49_code}")
        print(f"Name: {country.country_name}")
    else:
        print("Country not found!")

Country found!
Code: 995
Name: Notebook Test Country


In [ ]:
## 3. UPDATE – Modify Records

Update an existing country using SQLAlchemy ORM.

We retrieve the record, modify its attribute, and call `session.commit()` to save the changes.

In [7]:
with Session(engine) as session:

    country = session.get(Country, "995")

    if country:
        country.country_name = "Updated Notebook Country"

        session.commit()

        print("Country updated successfully!")
        print(f"New Name: {country.country_name}")
    else:
        print("Country not found!")

Country updated successfully!
New Name: Updated Notebook Country


In [ ]:
### 3.1 Verify Updated Record

Retrieve the country again to confirm that the updated value was saved.

In [8]:
with Session(engine) as session:

    country = session.get(Country, "995")

    if country:
        print(f"Code: {country.m49_code}")
        print(f"Updated Name: {country.country_name}")

Code: 995
Updated Name: Updated Notebook Country


In [ ]:
## 4. DELETE – Remove Records

Delete the test country from the database using `session.delete()`.

Call `session.commit()` to permanently save the deletion.

In [9]:
with Session(engine) as session:

    country = session.get(Country, "995")

    if country:
        session.delete(country)
        session.commit()

        print("Country deleted successfully!")
    else:
        print("Country not found!")

Country deleted successfully!


In [ ]:
### 4.1 Verify Deleted Record

Check whether the test country still exists in the database.

In [10]:
with Session(engine) as session:

    country = session.get(Country, "995")

    if country is None:
        print("Verified: Country was deleted.")
    else:
        print("Country still exists!")

Verified: Country was deleted.


In [ ]:
## 5. Transactions – COMMIT & ROLLBACK

A transaction is a group of database operations treated as one unit.

- `commit()`: Permanently saves changes.
- `rollback()`: Cancels uncommitted changes.

In [11]:
with Session(engine) as session:

    new_country = Country(
        m49_code="994",
        country_name="Commit Test Country"
    )

    session.add(new_country)
    session.commit()

    print("COMMIT: Record saved successfully!")

COMMIT: Record saved successfully!


In [ ]:
### 5.1 ROLLBACK – Cancel Changes

Insert a temporary record, then use `rollback()` before committing.

The inserted record should not be saved.

In [12]:
with Session(engine) as session:

    new_country = Country(
        m49_code="993",
        country_name="Rollback Test Country"
    )

    session.add(new_country)
    session.rollback()

    print("ROLLBACK: Changes cancelled!")

ROLLBACK: Changes cancelled!


In [ ]:
### 5.2 Verify Transaction Results

Check that the committed record exists and the rolled-back record does not exist.

In [13]:
with Session(engine) as session:

    committed = session.get(Country, "994")
    rolled_back = session.get(Country, "993")

    print("994 exists:", committed is not None)
    print("993 exists:", rolled_back is not None)

994 exists: True
993 exists: False


In [ ]:
## 6. Error Handling – Handling Database Errors

SQLAlchemy may raise errors when database constraints are violated.

We will test a duplicate Primary Key and handle the error using:

- `try / except`
- `IntegrityError`
- `rollback()`

In [14]:
from sqlalchemy.exc import IntegrityError

with Session(engine) as session:

    try:
        duplicate_country = Country(
            m49_code="994",
            country_name="Another Test Country"
        )

        session.add(duplicate_country)
        session.commit()

    except IntegrityError as error:
        session.rollback()

        print("Integrity Error: Duplicate Primary Key!")
        print("Transaction rolled back successfully.")

Integrity Error: Duplicate Primary Key!
Transaction rolled back successfully.


In [ ]:
## 7. Cleanup – Remove Test Data

Delete the temporary test country used in the COMMIT and Error Handling examples.

This keeps the original project database clean.

In [15]:
with Session(engine) as session:

    country = session.get(Country, "994")

    if country:
        session.delete(country)
        session.commit()
        print("Test data cleaned successfully!")
    else:
        print("No test data found.")

Test data cleaned successfully!


In [ ]:
## 8. Summary – SQLAlchemy CRUD & Transactions

In this notebook, we practiced:

1. Database Connection
2. CREATE – INSERT records
3. READ – SELECT records
4. UPDATE – Modify records
5. DELETE – Remove records
6. COMMIT & ROLLBACK – Manage transactions
7. Error Handling – Handle database constraint violations

All operations were tested successfully using PostgreSQL and SQLAlchemy ORM.